[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-13-review-advanced.ipynb#scrollTo=a1f2b3c4)

---
# Day 13 · Review — Advanced MLflow Patterns and Best Practices
**certified-journeys / mlflow-certified** · Review · Advanced Patterns

> **Goal for today:** Consolidate advanced MLflow knowledge — parent/child run nesting, reproducibility tags, the champion/challenger promotion pattern, cross-experiment comparison, and three common anti-patterns to avoid in production.


In [ ]:
%pip install -q mlflow scikit-learn pandas numpy


## Step 1 · Parent/Child Run Nesting for Hyperparameter Sweeps

When tuning hyperparameters you want **one parent run** that represents the entire sweep and **one child run per trial**. This structure makes the MLflow UI show a single collapsed entry for the sweep, with trials nested underneath.

| Concept | Role |
|---|---|
| **Parent run** | Created with `mlflow.start_run()` — holds sweep-level metadata and the final best metrics |
| **Child run** | Created with `mlflow.start_run(nested=True)` inside the parent context |
| **`nested=True`** | The flag that makes MLflow set `mlflow.parentRunId` tag automatically |
| **Parent final metrics** | Logged after the loop — copy the best child's metrics up to parent for easy filtering |

**Why this matters:** Without nesting, a 50-trial grid search creates 50 top-level runs that pollute the experiment view. With nesting, the UI shows 1 parent row — expand to see trials.


In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Local SQLite backend — enables the Model Registry without a running server
mlflow.set_tracking_uri("sqlite:///day13_review.db")
mlflow.set_experiment("day-13-advanced-review")

# Load a multi-class dataset
wine = load_wine(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)

# Hyperparameter grid for the sweep
param_grid = [
    {"n_estimators": 10,  "max_depth": 3},
    {"n_estimators": 50,  "max_depth": 5},
    {"n_estimators": 100, "max_depth": None},
]

best_acc = -1
best_params = None
best_run_id = None

# Parent run — represents the entire sweep
with mlflow.start_run(run_name="rf-sweep-parent") as parent_run:
    mlflow.set_tag("sweep.type", "grid")
    mlflow.set_tag("sweep.n_trials", len(param_grid))

    for i, params in enumerate(param_grid):
        # Child run — one per trial, nested=True links it to the parent
        with mlflow.start_run(run_name=f"trial-{i+1}", nested=True) as child_run:
            clf = RandomForestClassifier(**params, random_state=42)
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)
            acc = accuracy_score(y_test, preds)
            f1  = f1_score(y_test, preds, average="weighted")

            # Log all trial-level info on the child run
            mlflow.log_params(params)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_weighted", f1)
            mlflow.sklearn.log_model(clf, artifact_path="model")

            if acc > best_acc:
                best_acc   = acc
                best_params = params
                best_run_id = child_run.info.run_id

    # After loop: log best results on the PARENT so it summarises the sweep
    mlflow.log_metric("best_accuracy", best_acc)
    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})
    mlflow.set_tag("best_child_run_id", best_run_id)

parent_run_id = parent_run.info.run_id
print(f"Parent run id  : {parent_run_id}")
print(f"Best child run : {best_run_id}")
print(f"Best accuracy  : {best_acc:.4f}")
print(f"Best params    : {best_params}")


**What just happened?**
- **`nested=True`** caused MLflow to automatically set the `mlflow.parentRunId` tag on each child run — this is how the UI knows to collapse them.
- **Parent holds sweep-level metrics** (best accuracy, best params) — this makes filtering across many sweeps trivial because you never have to dig into children.
- **Children hold trial-level artifacts** — each has its own logged model, so you can load any trial's model by its `run_id` without guessing which artifact path belongs to the sweep.


## Step 2 · Reproducibility Tags with `mlflow.set_tags()`

Metrics and params capture *what* was trained. **Tags** capture *why it's reproducible* — the provenance metadata needed to recreate a result exactly.

| Tag key (convention) | Value | Purpose |
|---|---|---|
| `git.commit` | SHA from `git rev-parse HEAD` | Pinpoints exact code state |
| `git.branch` | Branch name | Tracks experiment lineage |
| `data.version` | Dataset hash or DVC tag | Ensures same data was used |
| `data.rows` | Integer | Quick sanity check |
| `env.mlflow_version` | `mlflow.__version__` | Guards against API drift |
| `env.sklearn_version` | `sklearn.__version__` | Guards against model drift |

> **Rule of thumb:** If a colleague clones the repo at the tagged commit and uses the tagged dataset, they should be able to reproduce the run's numbers exactly.


In [ ]:
import subprocess
import hashlib
import sklearn

def get_git_commit():
    """Return current HEAD SHA or 'unknown' if not in a git repo."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "unknown"  # Colab notebooks often aren't in a git repo

def get_data_hash(df: pd.DataFrame) -> str:
    """Stable hash of a DataFrame's values — cheap reproducibility signal."""
    raw = pd.util.hash_pandas_object(df, index=True).values.tobytes()
    return hashlib.sha256(raw).hexdigest()[:16]

# Build the reproducibility tag dictionary once — reuse across all runs
repro_tags = {
    "git.commit":         get_git_commit(),
    "git.branch":         "main",              # hard-coded here; in CI, read from env
    "data.version":       get_data_hash(wine.data),
    "data.rows":          str(len(wine.data)),
    "data.source":        "sklearn.datasets.load_wine",
    "env.mlflow_version": mlflow.__version__,
    "env.sklearn_version": sklearn.__version__,
}

print("Reproducibility tags:")
for k, v in repro_tags.items():
    print(f"  {k:28s} = {v}")

# Attach tags to the best child run we found in Step 1
client = MlflowClient()
client.set_tags(best_run_id, repro_tags)

# Verify the tags were written
run_data = client.get_run(best_run_id).data
print(f"\nTags on best child run (sample):")
for k in ["git.commit", "data.version", "env.mlflow_version"]:
    print(f"  {k}: {run_data.tags.get(k, '(missing)')}")


**What just happened?**
- **`mlflow.set_tags(dict)`** is a bulk-set call — much cleaner than calling `mlflow.set_tag(k, v)` repeatedly.
- **`pd.util.hash_pandas_object`** is a fast, stable hash of all DataFrame values — if the data changes even by one row, the hash changes.
- We used **`client.set_tags(run_id, tags)`** to retroactively attach tags to an already-completed run — you don't have to be inside an active `start_run` context.


## Step 3 · Model Aliases and the Champion/Challenger Promotion Pattern

The **champion/challenger** pattern is the industry-standard approach to safe model promotion:

| Role | Alias | Description |
|---|---|---|
| **Champion** | `champion` | Currently serving production traffic |
| **Challenger** | `challenger` | New candidate — shadow-testing or A/B |
| **Previous** | `previous` | Last champion — instant rollback target |

**Promotion flow:**
1. Train new model → register as `challenger`
2. Run offline evaluation: challenger must beat champion on key metrics
3. If challenger wins: `previous ← champion`, `champion ← challenger`
4. If challenger loses: archive it, champion stays

Aliases are **mutable pointers** — your serving code always loads `models:/wine-rf@champion` and never needs to change, even when the champion version changes.


In [ ]:
MODEL_NAME = "wine-classifier"

# --- Register the best child run's model as the initial champion ---
champion_uri = f"runs:/{best_run_id}/model"
mv_champion = mlflow.register_model(model_uri=champion_uri, name=MODEL_NAME)
print(f"Registered v{mv_champion.version} as initial model")

client.set_registered_model_alias(MODEL_NAME, "champion", mv_champion.version)
client.update_model_version(
    name=MODEL_NAME,
    version=mv_champion.version,
    description=f"Initial champion. accuracy={best_acc:.4f}, params={best_params}",
)
print(f"Set alias 'champion' → v{mv_champion.version}")

# --- Train a challenger model ---
with mlflow.start_run(run_name="challenger-candidate") as chal_run:
    from sklearn.ensemble import GradientBoostingClassifier
    gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42)
    gb.fit(X_train, y_train)
    chal_preds = gb.predict(X_test)
    chal_acc = accuracy_score(y_test, chal_preds)

    mlflow.log_params({"n_estimators": 200, "learning_rate": 0.05})
    mlflow.log_metric("accuracy", chal_acc)
    mlflow.set_tags(repro_tags)  # always attach reproducibility tags
    mlflow.sklearn.log_model(gb, artifact_path="model")
    chal_run_id = chal_run.info.run_id

# Register challenger
mv_challenger = mlflow.register_model(f"runs:/{chal_run_id}/model", MODEL_NAME)
client.set_registered_model_alias(MODEL_NAME, "challenger", mv_challenger.version)
print(f"\nChallenger v{mv_challenger.version} accuracy: {chal_acc:.4f}")
print(f"Champion   v{mv_champion.version} accuracy:  {best_acc:.4f}")

# --- Promotion decision ---
IMPROVEMENT_THRESHOLD = 0.001  # challenger must beat champion by at least 0.1%

if chal_acc > best_acc + IMPROVEMENT_THRESHOLD:
    print("\nChallenger wins — promoting to champion.")
    # Save previous champion before overwriting alias
    client.set_registered_model_alias(MODEL_NAME, "previous", mv_champion.version)
    # Atomically promote challenger to champion
    client.set_registered_model_alias(MODEL_NAME, "champion", mv_challenger.version)
    client.delete_registered_model_alias(MODEL_NAME, "challenger")
    print(f"  champion  → v{mv_challenger.version}")
    print(f"  previous  → v{mv_champion.version}")
else:
    print("\nChallenger did NOT beat champion — keeping current champion.")
    # Archive challenger so registry stays clean
    client.transition_model_version_stage(
        name=MODEL_NAME, version=mv_challenger.version,
        stage="Archived",
    )
    print(f"  Challenger v{mv_challenger.version} archived.")

# Show current alias map
print("\n=== Current aliases ===")
rm = client.get_registered_model(MODEL_NAME)
for alias, ver in rm.aliases.items():
    print(f"  @{alias} → v{ver}")


**What just happened?**
- **`set_registered_model_alias`** is idempotent — calling it again on the same alias name just moves the pointer, so the champion rotation is a single atomic update.
- **`delete_registered_model_alias`** is used to remove the `challenger` alias after promotion — keeping stale aliases causes confusion in the UI.
- **`rm.aliases`** is a `{alias: version_string}` dict — a quick way to audit which versions are currently tagged as champions, challengers, etc.
- **`IMPROVEMENT_THRESHOLD`** prevents noise-level fluctuations from triggering unnecessary promotions.


## Step 4 · Cross-Experiment Comparison with `MlflowClient.search_runs`

The MLflow UI's **Comparison view** lets you select runs and chart them side-by-side. Programmatically, `search_runs` is the equivalent — essential for automated reporting or CI gates that compare across experiments.

| `search_runs` parameter | Purpose |
|---|---|
| `experiment_names` | List of experiment names to search across |
| `filter_string` | SQL-like filter, e.g. `"metrics.accuracy > 0.9"` |
| `order_by` | Sort, e.g. `["metrics.accuracy DESC"]` |
| `max_results` | Cap on returned runs |
| Return value | `pd.DataFrame` — each row is a run |


In [ ]:
# Create a second experiment so we have something to compare across
mlflow.set_experiment("day-13-comparison-experiment")
with mlflow.start_run(run_name="logreg-baseline"):
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline

    pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=500))])
    pipe.fit(X_train, y_train)
    lr_acc = accuracy_score(y_test, pipe.predict(X_test))
    mlflow.log_metric("accuracy", lr_acc)
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.set_tag("experiment_group", "baselines")

print(f"LogisticRegression accuracy: {lr_acc:.4f}")

# ---- Cross-experiment search ----
runs_df = mlflow.search_runs(
    experiment_names=["day-13-advanced-review", "day-13-comparison-experiment"],
    filter_string="metrics.accuracy > 0.5",   # exclude failed/unfinished runs
    order_by=["metrics.accuracy DESC"],
    max_results=20,
)

# Select the columns most useful for comparison
cols = ["run_id", "experiment_id", "tags.mlflow.runName",
        "metrics.accuracy", "params.n_estimators", "params.model_type"]
available = [c for c in cols if c in runs_df.columns]
print("\nCross-experiment comparison (top 6):")
print(runs_df[available].head(6).to_string(index=False))


**What just happened?**
- **`mlflow.search_runs(experiment_names=[...])`** accepts a list — this is how you query across multiple experiments in one call without iterating.
- The result is a **`pandas.DataFrame`** with columns like `metrics.accuracy`, `params.n_estimators`, `tags.mlflow.runName` — ready for plotting or exporting to a report.
- **`filter_string`** uses MLflow's SQL-like DSL — you can combine conditions with `AND`/`OR` and filter on tags, params, or metrics.
- This is the programmatic equivalent of clicking "Compare" in the MLflow UI Experiments view.


## Step 5 · Three Critical Anti-Patterns to Avoid

These are the top mistakes that appear in production MLflow codebases:

| # | Anti-pattern | Why it's bad | Correct approach |
|---|---|---|---|
| 1 | **Logging outside a run** | Values go into `default` experiment with no traceability | Always wrap logging in `with mlflow.start_run():` |
| 2 | **Mutable artifacts** | Re-uploading to the same path silently overwrites; no version history | Log to a new run or use the Model Registry for versioning |
| 3 | **Skipping signatures** | Model can serve wrong input shapes; no schema validation at serving time | Always pass `input_example` or build `mlflow.models.infer_signature` manually |

The code below **demonstrates** each anti-pattern and then shows the correct equivalent — study the diffs carefully.


In [ ]:
from mlflow.models import infer_signature

# =========================================================
# ANTI-PATTERN 1: Logging outside a run
# =========================================================
# BAD — this logs to the default experiment with an auto-generated run
# and there is no context to know WHAT it belongs to:
#   mlflow.log_metric("accuracy", 0.95)   # <- DO NOT DO THIS
#
# CORRECT — always be explicit:
mlflow.set_experiment("day-13-advanced-review")
with mlflow.start_run(run_name="anti-pattern-demo") as demo_run:

    # =====================================================
    # ANTI-PATTERN 2: Logging a mutable artifact
    # =====================================================
    # BAD — writing to a fixed filename and re-logging it changes the artifact
    # silently. Any previous version is gone with no history:
    #   with open("report.txt", "w") as f: f.write("v1 content")
    #   mlflow.log_artifact("report.txt")   # <- first log
    #   with open("report.txt", "w") as f: f.write("v2 content")
    #   mlflow.log_artifact("report.txt")   # <- SILENTLY OVERWRITES v1
    #
    # CORRECT — version the artifact name, or start a new run:
    import tempfile, os
    with tempfile.NamedTemporaryFile(mode="w", suffix="_v1.txt", delete=False) as f:
        f.write("Run notes: first training pass.\nAccuracy = 0.94")
        tmp_path = f.name

    # Use a versioned artifact_path so each write is distinct
    mlflow.log_artifact(tmp_path, artifact_path="reports/v1")
    os.unlink(tmp_path)  # clean up local temp file

    # =====================================================
    # ANTI-PATTERN 3: Skipping model signature
    # =====================================================
    # BAD — logging a model without a signature:
    #   mlflow.sklearn.log_model(clf, "model")   # <- no signature
    # The serving endpoint will accept ANY input shape, causing silent failures.
    #
    # CORRECT — always include a signature:
    from sklearn.ensemble import RandomForestClassifier
    demo_clf = RandomForestClassifier(n_estimators=10, random_state=0)
    demo_clf.fit(X_train, y_train)
    demo_preds = demo_clf.predict(X_test)

    # infer_signature captures input columns/dtypes and output schema
    signature = infer_signature(X_train, demo_preds)

    mlflow.sklearn.log_model(
        sk_model=demo_clf,
        artifact_path="model-with-signature",
        signature=signature,          # schema validation at serve time
        input_example=X_train.head(2),  # sample for docs / UI preview
    )
    mlflow.log_metric("accuracy", accuracy_score(y_test, demo_preds))

    demo_run_id = demo_run.info.run_id

# Show the signature that was saved
model_info = mlflow.sklearn.load_model(f"runs:/{demo_run_id}/model-with-signature")
loaded_sig = mlflow.models.get_model_info(f"runs:/{demo_run_id}/model-with-signature").signature
print("Saved signature:")
print(f"  inputs : {loaded_sig.inputs}")
print(f"  outputs: {loaded_sig.outputs}")


**What just happened?**
- **Anti-pattern 1 fix:** wrapping all logging in `with mlflow.start_run():` guarantees traceability — every metric, param, and artifact is associated with a specific run in a specific experiment.
- **Anti-pattern 2 fix:** using a versioned `artifact_path` means each write lands in a distinct URI (`mlruns/<id>/artifacts/reports/v1/`), so no prior version is overwritten.
- **`infer_signature(X, y_pred)`** inspects the DataFrame schema and prediction array to build a typed contract — MLflow Serving enforces this contract and returns a 400 error if a client sends wrong column names or dtypes.


## Step 6 · Reviewing Parent/Child Relationships Programmatically

The MLflow Comparison view in the UI is powerful for ad-hoc analysis. For programmatic CI pipelines or automated reports, you need to reconstruct the parent/child relationship using the `mlflow.parentRunId` tag.

The tag name `mlflow.parentRunId` is always set automatically by MLflow when `nested=True` — it is a system tag, not a user tag.


In [ ]:
# Retrieve all child runs of our Step 1 parent sweep
children = mlflow.search_runs(
    experiment_names=["day-13-advanced-review"],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run_id}'",
    order_by=["metrics.accuracy DESC"],
)

print(f"Found {len(children)} child runs for parent {parent_run_id[:8]}…")

child_cols = ["run_id", "tags.mlflow.runName", "metrics.accuracy",
              "params.n_estimators", "params.max_depth"]
available_child = [c for c in child_cols if c in children.columns]
print(children[available_child].to_string(index=False))

# Confirm the parent run's stored 'best_accuracy' matches the child we found
parent_data = client.get_run(parent_run_id).data
print(f"\nParent stored best_accuracy : {parent_data.metrics.get('best_accuracy', 'N/A'):.4f}")
if len(children) > 0:
    top_child_acc = children.iloc[0]["metrics.accuracy"]
    print(f"Top child accuracy          : {top_child_acc:.4f}")
    print(f"Match: {abs(parent_data.metrics.get('best_accuracy', 0) - top_child_acc) < 1e-6}")


**What just happened?**
- **`tags.mlflow.parentRunId = '...'`** in the filter string is the programmatic equivalent of expanding the parent row in the MLflow UI to see its children.
- We confirmed the **parent's stored `best_accuracy`** matches the top child — this is the discipline of always promoting results upward so the parent is self-describing.
- This pattern scales: a CI job can `search_runs(filter_string=f"tags.mlflow.parentRunId='{sweep_run_id}'")` to collect all trial metrics without knowing individual run IDs in advance.


In [ ]:
# Challenge: Build a sweep auditor
#
# Write a function `audit_sweep(parent_run_id, experiment_name)` that:
#   1. Fetches all child runs of the given parent (using search_runs + parentRunId tag)
#   2. Finds the child with the highest 'accuracy' metric
#   3. Checks that the parent run has a 'best_accuracy' metric that matches
#      the best child's accuracy (within 1e-6 tolerance)
#   4. Returns a dict with keys:
#        - 'n_children': number of child runs found
#        - 'best_run_id': run_id of the top child
#        - 'best_accuracy': float
#        - 'parent_consistent': bool (True if parent best_accuracy matches)
#
# Hints:
#   - mlflow.search_runs(experiment_names=[...], filter_string=...) returns a DataFrame
#   - client.get_run(run_id).data.metrics['best_accuracy']
#   - DataFrame column for accuracy: 'metrics.accuracy'
#   - Handle the case where the parent has no 'best_accuracy' metric

def audit_sweep(parent_run_id, experiment_name):
    # Your solution here
    pass

# Test it (uncomment after implementing):
# result = audit_sweep(parent_run_id, "day-13-advanced-review")
# print(result)


---
## Day 13 key concepts recap

| Concept | What to remember |
|---|---|
| `nested=True` in `start_run` | Links child to parent via `mlflow.parentRunId` tag; collapses sweep in UI |
| Parent holds best metrics | Copy best trial metrics up to parent so sweeps are filterable without opening children |
| `mlflow.set_tags(dict)` | Bulk-set reproducibility tags: git commit, data hash, library versions |
| Champion/challenger aliases | `@champion`, `@challenger`, `@previous` — serving code uses aliases, never version numbers |
| `delete_registered_model_alias` | Clean up stale aliases after promotion to avoid confusion |
| `search_runs(experiment_names=[...])` | Cross-experiment query; returns a DataFrame; filter with SQL-like DSL |
| Anti-pattern: log outside run | All `log_*` calls must be inside `with mlflow.start_run():` |
| Anti-pattern: mutable artifacts | Version artifact paths (e.g. `reports/v1/`) — never overwrite an existing path |
| Anti-pattern: no signature | Always pass `input_example` or `infer_signature` to `log_model` |

> **Tip:** Parent/child runs are the cleanest way to track hyperparameter sweeps — the parent holds the final best metrics; child runs hold the individual trial results.

---
## What's next
**Day 14** → Capstone — Full MLOps Pipeline: Train → Track → Register → Serve. You will combine everything from the course into a single reproducible pipeline.

Mark Day 13 complete in your [tracker](../index.html).
